# Data Center Cooling Digital Twin & Control Architecture

Here we explore a hardware/software-in-the-Loop simulation of a data center thermal management system. We will separate the physical process (the "plant"), the safety interlocks (the PLC), and the supervisory control algorithms (PID/MPC) into distinct networked components, mimicking a real-world industrial architecture.

PLCs are purpose-built industrial computers. While a standard PC or Raspberry Pi might crash, reboot for updates, or delay a process because the CPU is busy, a PLC is fundamentally different. PLCs operate on strict, predictable scan cycles. A PLC reads inputs, executes its logic, and updates outputs in precisely the same amount of time (often single-digit milliseconds), ensuring real-time reliability.
They are physically designed to survive brutal environments. They withstand extreme temperature fluctuations, high humidity, intense physical vibration, and heavy electromagnetic interference from massive factory motors. PLCs are designed to run 24/7/365 for decades without being turned off. If a fault occurs, they are programmed to fail into a known, safe state to prevent catastrophic industrial accidents. Due to their reliability, PLCs are the unseen backbone of modern civilization. Modern industrial control systems are highly distributed. Our simulation consists of three primary nodes:

1. **The Digital Twin (Plant Simulator):** A Python script (`digital_twin.py`) that models the thermodynamics of the server hall. It acts as our virtual sensors and actuators.
2. **The Programmable Logic Controller (PLC):** An OpenPLC Runtime v4 container executing IEC 61131-3 Ladder Logic. It acts as the local I/O layer, handling high-speed safety interlocks (e.g., thermal shutdown alarms).
3. **The Supervisory Controller:** A Python script (`mpc.py` or `pid.py`) representing a Building Management System (BMS). It computes the optimal control effort (fan speed) to maintain a desired temperature setpoint.

These three components exchange data in real-time over the network using Modbus TCP.

Modbus is one of the oldest and most ubiquitous industrial communication protocols. Originally developed by Modicon in 1979 for serial lines, Modbus TCP adapts the protocol for modern Ethernet networks (typically on Port `5020` or `502`). Modbus uses a client/server architecture. In our setup, the OpenPLC container is the Server, hosting the memory maps. The Python scripts (`digital_twin.py` and `mpc.py`) are Clients that read from and write to the Server.

Modbus defines four fundamental data tables. OpenPLC maps these to specific IEC 61131-3 memory addresses:

| Modbus Table | Type | Read/Write | Modicon Address | OpenPLC Mapping | Use Case in this Lab |
| --- | --- | --- | --- | --- | --- |
| **Coils** | Boolean | R/W | `00001+` | `%QX` | PLC Overheat Alarm (`%QX0.0`) |
| **Discrete Inputs** | Boolean | Read-Only | `10001+` | `%IX` | Physical push-buttons (Not used) |
| **Input Registers** | 16-bit Int | Read-Only | `30001+` | `%IW` | Analog sensors (Read-only) |
| **Holding Registers** | 16-bit Int | R/W | `40001+` | `%MW` / `%QW` | Temp/Power (`%MW0`, `%MW1`), Fan Speed (`%QW0`) |







*Note on Data Types: Modbus natively only understands 16-bit integers. To transmit floating-point data (like $29.50^\circ\text{C}$), we use fixed-point scaling. The Digital Twin multiplies the temperature by $100$ (sending `2950`), and the Controller divides by $100$ upon reading it.*

The `digital_twin.py` script computes the air-side energy balance of the data center. The physics are evaluated at discrete time steps $k$.

The cooling air is driven by CRAC (Computer Room Air Conditioning) unit fans. The fan speed command $u_k \in [0, 100]\%$ dictates the volumetric flow rate $\dot{V}_k$. We include a small leakage/convection term $\dot{V}_{\text{leak}}$ so that flow is not zero when the fans are off.

$$\dot{m}_k = \left( V_{\max} \frac{u_k}{100} + \dot{V}_{\text{leak}} \right) \rho_{\text{air}}$$

Where:

* $\dot{m}_k$ is the air mass flow rate ($\text{kg/s}$)
* $V_{\max} = 50.0 \text{ m}^3/\text{s}$ (Maximum volumetric flow)
* $\dot{V}_{\text{leak}} = 0.5 \text{ m}^3/\text{s}$ (Leakage flow)
* $\rho_{\text{air}} = 1.225 \text{ kg/m}^3$ (Density of air)

The servers dissipate heat $\dot{Q}_{\text{IT},k}$ ($\text{kW}$) into the passing air stream. Based on the sensible heat transfer equation $\dot{Q} = \dot{m} c_p \Delta T$, the steady-state equilibrium outlet temperature $T_{\text{eq},k}$ is:

$$T_{\text{eq},k} = T_{\text{amb}} + \frac{\dot{Q}_{\text{IT},k}}{\dot{m}_k c_p}$$

Where:

* $T_{\text{amb}} = 20.0^\circ\text{C}$ (Inlet cooling air temperature)
* $c_p = 1.006 \text{ kJ/(kg}\cdot\text{K)}$ (Specific heat of air)

A real data center does not change temperature instantly. The servers, racks, and air volume possess thermal mass. We model this as a first-order low-pass filter (a lumped capacitance model), governed by the continuous-time differential equation:

$$\tau \frac{dT(t)}{dt} = T_{\text{eq}}(t) - T(t)$$

Where $\tau$ is the thermal time constant of the system ($\tau = \frac{m c_p}{h A}$).

To evaluate this within a digital twin or PLC at discrete computational time steps ($\Delta t$), we approximate the continuous derivative using the Forward Euler method:

$$\frac{dT(t)}{dt} \approx \frac{T_{k+1} - T_k}{\Delta t}$$

Substituting this approximation back into our physical model gives:

$$\tau \left( \frac{T_{k+1} - T_k}{\Delta t} \right) = T_{\text{eq},k} - T_k$$

Multiplying by $\frac{\Delta t}{\tau}$ and rearranging to solve for the temperature at the next time step ($T_{k+1}$):

$$T_{k+1} = T_k + \frac{\Delta t}{\tau} \left( T_{\text{eq},k} - T_k \right)$$

By defining the thermal inertia factor as $\alpha = \frac{\Delta t}{\tau}$, we arrive at our final discrete-time update equation:

$$T_{k+1} = T_k + \alpha \left( T_{\text{eq},k} - T_k \right)$$

Where $\alpha \in (0, 1]$ represents the ratio of the computational time step to the physical thermal time constant. For instance, with $\alpha = 0.2$, the server temperature covers 20% of the difference toward the equilibrium temperature in a single step. For numerical and physical stability, $\Delta t$ must be chosen such that $\alpha \le 1$, preventing the model from overshooting equilibrium and oscillating.

The objective of our controllers is to modulate the fan speed $u_k$ to maintain the outlet temperature at a target setpoint $T_{\text{set}}$, while mitigating stochastic disturbances (fluctuating IT load $\dot{Q}_{\text{IT}}$).

The standard Proportional-Integral-Derivative (PID) controller reacts to the instantaneous error $e_k = T_k - T_{\text{set}}$:

$$u_k = K_p e_k + K_i \sum_{j=0}^{k} e_j \Delta t + K_d \frac{e_k - e_{k-1}}{\Delta t}$$

**Challenge with PID:** The plant physics are highly nonlinear. The sensitivity of the temperature to the fan speed $\left( \frac{\partial T}{\partial u} \right)$ changes drastically depending on the current fan speed. A fixed-gain PID controller that is well-tuned for $100\%$ fan speed may become violently unstable at $20\%$ fan speed.

To handle the severe nonlinearity and enforce strict physical constraints, we implement model predictive control (MPC). At each time step $k$, the MPC algorithm solves a finite-horizon Open-Loop Optimal Control Problem (OCP) over a prediction horizon of $N$ steps.

Disturbance Estimation: If the true IT load is unknown, we estimate it by inverting the temperature transition from the previous time step:

$$\hat{Q}_{\text{IT},k} = \left( \frac{T_k - T_{k-1}}{\alpha} + T_{k-1} - T_{\text{amb}} \right) \dot{m}_{k-1} c_p$$

The Optimal Control Problem: The MPC minimizes a cost function $J$ over the sequence of future control actions $U = [u_{k\vert{}k}, u_{k+1\vert{}k}, \dots, u_{k+N-1\vert{}k}]$.

$$\min_{U} J(U) = \sum_{j=0}^{N-1} \left( \underbrace{w_T (T_{k+j+1} - T_{\text{set}})^2}_{\text{Tracking Error}} + \underbrace{w_u (u_{k+j} - u_{\text{ss}})^2}_{\text{Control Effort}} + \underbrace{w_{\Delta u} (\Delta u_{k+j})^2}_{\text{Slew Rate}} + \underbrace{w_S S_{k+j+1}^2}_{\text{Safety Penalty}} \right)$$

Subject to the physical dynamics and constraints:

* **System Dynamics:** $T_{k+j+1} = f(T_{k+j}, u_{k+j}, \hat{Q}_{\text{IT},k})$
* **Actuator Bounds:** $u_{\min} \le u_{k+j} \le u_{\max}$
* **Slew Rate:** $\Delta u_{k+j} = u_{k+j} - u_{k+j-1}$
* **Steady-State Target:** $u_{\text{ss}}$ is the analytical fan speed required to hold $T_{\text{set}}$ at the estimated load $\hat{Q}_{\text{IT},k}$.
* **Soft Safety Barrier:** $S = \max(0, T - T_{\text{alarm}})$ penalizes the controller exponentially if it approaches the PLC's hard $35.0^\circ\text{C}$ shutdown threshold.

Once the optimal sequence $U^*$ is found, the controller applies only the first control move ($u_k = u_{k\vert{}k}^*$) to the plant, and the optimization is repeated at the next time step. This is known as Receding Horizon Control.

## Appendix A: Programmable Logic Controllers (PLCs)

A programmable logic controller (PLC) is a ruggedized, deterministic industrial computer adapted for the control of manufacturing processes, robotics, or any activity that requires high reliability, ease of programming, and process fault diagnosis. Unlike consumer PCs, PLCs operate strictly in real-time, executing a continuous scan cycle: read inputs, execute logic, write outputs, and repeat (often on the scale of milliseconds). Modern PLCs adhere to the IEC 61131-3 international standard, which normalizes how PLCs are programmed across different manufacturers (e.g., Siemens, Allen-Bradley, Beckhoff, and open-source platforms like OpenPLC).

IEC 61131-3 uses a specific syntax to map variables to hardware memory locations. The syntax follows the format: `%[Prefix][Size][Address]`

*   **Prefix (Location Type):**
    *   `I` : Physical Input (read-only from the PLC's perspective)
    *   `Q` : Physical Output (read/write from the PLC)
    *   `M` : Internal Memory (virtual registers)
*   **Size:**
    *   `X` : Bit (1 bit) — often omitted in modern tooling, but implies a boolean.
    *   `B` : Byte (8 bits)
    *   `W` : Word (16 bits)
    *   `D` : Double Word (32 bits)
    *   `L` : Long Word (64 bits)

**Examples:**
*   `%QX0.0` : Output, Bit size, Byte 0, Bit 0. (Used for our `overheat_alarm` coil).
*   `%MW0` : Memory, Word size, Word offset 0. (Used for our `sensor_temp` holding register).
*   `%IX1.2` : Input, Bit size, Byte 1, Bit 2.

PLCs are strongly typed. Common data types include:
*   `BOOL`: Boolean ($0$ or $1$). Used for alarms, switches, and relays.
*   `INT` / `UINT`: 16-bit signed/unsigned integers. Native to Modbus registers.
*   `REAL`: 32-bit floating-point number. (Note: Transmitting `REAL` over Modbus requires splitting the 32 bits across two 16-bit registers, which is why we often use `INT` with fixed-point scaling instead).
*   `TIME`: Duration (e.g., `T#2s` for a 2-second timer delay).

The standard defines five programming languages, allowing engineers to choose the best paradigm for the task:

1.  **Ladder Diagram (LD):** The most common graphical language, designed to mimic legacy electrical relay schematics. Power flows from a left "rail" through normally-open (`| |`) or normally-closed (`|/|`) contacts to energize coils (`( )`) on the right. Excellent for interlocking safety logic.
2.  **Function Block Diagram (FBD):** A graphical language where signals flow through blocks (like PID, timers, or math functions) wired together. It closely resembles Simulink and is great for continuous process control.
3.  **Structured Text (ST):** A high-level, text-based language similar to Pascal or C. It is highly efficient for complex math, loops (`FOR`, `WHILE`), and state machine logic that is tedious to draw graphically. Under the hood, OpenPLC's compiler (STruC++) converts graphical languages into ST before compiling them to C++.
4.  **Sequential Function Chart (SFC):** A graphical flow-chart language used for coordinating sequences of operations (States and Transitions).
5.  **Instruction List (IL):** A low-level assembly-like language (deprecated in modern updates to the standard).

## Appendix B: Modbus Protocol

Developed in 1979 by Modicon (now Schneider Electric), Modbus has become the *de facto* standard communication protocol in industrial automation due to its simplicity, royalty-free nature, and robust design. 

*   **Modbus RTU (Remote Terminal Unit):** The original serial protocol transmitted over RS-232 or RS-485 cables. Data is transmitted in binary.
*   **Modbus TCP/IP:** Modbus packets encapsulated within TCP/IP wrappers, allowing them to traverse standard Ethernet networks and the Internet (typically using port `502` or `5020`). This is the variant used in our digital twin architecture.

Modbus strictly follows a **Client/Server** (historically Master/Slave) architecture:
*   **Server (The PLC):** The device that holds the data. It passively listens for requests on the network. It does not initiate communication.
*   **Client (The Python Scripts):** The device (or scripts) that initiates communication. The client requests data to read or commands data to write. 

When a Client sends a request to a Server, it includes a Function Code (FC) that dictates what action the Server should perform on its memory tables.

| Function Code (Hex/Dec) | Action | Target Modbus Table |
| :--- | :--- | :--- |
| `01` (`0x01`) | Read Coils | Coils (`%QX`) |
| `02` (`0x02`) | Read Discrete Inputs | Discrete Inputs (`%IX`) |
| `03` (`0x03`) | Read Holding Registers | Holding Registers (`%MW` / `%QW`) |
| `04` (`0x04`) | Read Input Registers | Input Registers (`%IW`) |
| `05` (`0x05`) | Write Single Coil | Coils (`%QX`) |
| `06` (`0x06`) | Write Single Register | Holding Registers (`%MW` / `%QW`) |
| `15` (`0x0F`) | Write Multiple Coils | Coils (`%QX`) |
| `16` (`0x10`) | Write Multiple Registers| Holding Registers (`%MW` / `%QW`) |

*In our `digital_twin.py` script, `client.write_register()` implicitly uses FC 06, while `client.read_holding_registers()` uses FC 03.*

**The 16-bit Limit:**
The fundamental limitation of Modbus is that its registers are strictly 16-bit integers (capable of representing values from $0$ to $65,535$ unsigned, or $-32,768$ to $32,767$ signed). 

**Handling Floating Point Numbers:**
Because Modbus lacks native floating-point types, engineers use two common workarounds:
1.  **Fixed-Point Scaling (Used in this Lab):** Multiply the float by a factor of $10$ or $100$ before transmission, and divide by the same factor upon receipt. For example, $29.50^\circ\text{C} \times 100 = 2950$. The integer $2950$ fits perfectly in a 16-bit register.
2.  **IEEE 754 Floating-Point Split:** A 32-bit `REAL` float is split across two consecutive 16-bit Holding Registers. The Client must read both registers and concatenate the bits back into a 32-bit float. This introduces the risk of "Word Swap" or "Byte Swap" errors if the Client and Server use different endianness (byte ordering).

In PLC and Modbus terminology, memory is organized based on how data is interacted with (read vs. write) and its data type (boolean vs. numerical).

* **Inputs (Discrete Inputs):** These are read-only boolean (1-bit) values. They represent physical digital signals coming *into* the PLC from the real world, such as a physical push-button being pressed, a limit switch being triggered on a conveyor belt, or a proximity sensor detecting an object.
* **Coils (Discrete Outputs):** These are read/write boolean (1-bit) values. The term "coil" is a historical holdover from early automation when controllers energized physical electromagnetic relay coils to close a switch. Today, a coil represents a digital output command, such as turning on a fan, opening a valve, or sounding an alarm.
* **Registers:** These are 16-bit memory locations used to store numerical data rather than simple on/off states.
* **Input Registers** are read-only and store analog sensor data coming in (e.g., a continuous temperature reading, pressure level, or fluid flow rate).
* **Holding Registers** are read/write and are used to store configuration settings, setpoints, or output commands (e.g., commanding a variable frequency drive to run a motor at exactly 45.5 Hz, or storing a PID tuning parameter).